# 09 · Limpieza: Consolidados de asistencia por KAM 2026_anonimizado

Resumen de asistencia a eventos de 2026 de las empresas de cada KAM (asesor comercial). Una fila = una empresa de la cartera de un KAM.

- Entrada: `datos/originales/Consolidados de asistencia por KAM 2026_anonimizado.xlsx` (hoja `Hoja 1`)
- Salida: `datos/limpios/09_asistencia_por_kam_limpio.csv`

In [ ]:
from pathlib import Path

import pandas as pd

BASE = Path.cwd().parent
ORIGINALES = BASE / "datos" / "originales"
LIMPIOS = BASE / "datos" / "limpios"
LIMPIOS.mkdir(exist_ok=True)

## 1. Cómo viene la hoja
No es una tabla, sino **6 bloques uno debajo del otro, uno por KAM** (A, D, F, B, E, C). Cada bloque tiene:
1. Un título (`RESUMEN CONSOLIDADO DE ASISTENCIAS - BD KAM A`) y una fila de resumen: total de empresas, con asistencia, sin asistencia y total de asistentes.
2. A la izquierda (columnas 1 a 5), todas las empresas del KAM: estado de asistencia, eventos asistidos y total de asistentes.
3. A la derecha (columnas 7 a 9), solo las empresas **sin** asistencia, con la acción sugerida.

**Decisiones:**
- Se recorre la hoja y cada empresa toma el KAM del último título que apareció arriba.
- Se usa solo la tabla de la izquierda: ya incluye a las empresas sin asistencia (`No Asistió`). La de la derecha solo repite esas mismas empresas con la acción "Pendiente Convocatoria", así que se usa para **comprobar** y no se guarda aparte.
- Los subtítulos de cada bloque (`CONSOLIDADO GENERAL POR EMPRESA (BD …)`) traen el **nombre real** de cada KAM. Esa información **no se copia** a los datos limpios, que solo usan `KAM A`…`KAM F`, como el resto de las bases anonimizadas.

In [ ]:
crudo = pd.read_excel(ORIGINALES / "Consolidados de asistencia por KAM 2026_anonimizado.xlsx", sheet_name="Hoja 1", header=None)
assert crudo.shape == (506, 10), crudo.shape

col1 = crudo[1].astype("string").str.strip()
titulo = col1.str.extract(r"^RESUMEN CONSOLIDADO DE ASISTENCIAS - BD (KAM [A-Z])$", expand=False)
crudo["kam"] = titulo.ffill()  # cada fila hereda el KAM del bloque en el que está
print("Bloques encontrados:", titulo.dropna().tolist())
assert titulo.notna().sum() == 6

## 2. Tabla de empresas
Se toman las filas cuya columna 1 es un id de empresa (`EMP-0000`). Así quedan fuera, en un solo paso, títulos, encabezados, filas de resumen y filas vacías.

In [ ]:
es_empresa = col1.str.fullmatch(r"EMP-\d{4}", na=False)
df = crudo.loc[es_empresa, ["kam", 1, 3, 4, 5]].copy()
df.columns = ["kam", "id_empresa", "estado_asistencia", "eventos_asistidos", "total_asistentes"]
df["id_empresa"] = df["id_empresa"].str.strip()
df[["eventos_asistidos", "total_asistentes"]] = df[["eventos_asistidos", "total_asistentes"]].astype(int)
df["asistio"] = df["estado_asistencia"].map({"Asistió": True, "No Asistió": False})
assert df["asistio"].notna().all(), set(df["estado_asistencia"])
df = df.drop(columns="estado_asistencia").reset_index(drop=True)
df.head()

## 3. Validaciones
- Cada empresa aparece en un solo KAM.
- `asistio` coincide con tener al menos un evento asistido, y nunca hay menos asistentes que eventos.
- Los conteos por KAM coinciden con la fila de resumen de cada bloque.
- Las empresas de la tabla de la derecha (sin asistencia) son exactamente las que tienen `asistio = False` en su mismo KAM.

In [ ]:
assert df["id_empresa"].is_unique
assert (df["asistio"] == (df["eventos_asistidos"] > 0)).all()
assert (df["total_asistentes"] >= df["eventos_asistidos"]).all()

# Fila de resumen: está justo debajo de la fila "Total Empresas BD"
resumen = crudo.loc[col1.eq("Total Empresas BD").shift(1, fill_value=False), ["kam", 1, 2, 3, 4]]
resumen.columns = ["kam", "total_empresas", "con_asistencia", "sin_asistencia", "total_asistentes"]
resumen = resumen.set_index("kam").astype(int)
calculado = df.groupby("kam").agg(total_empresas=("id_empresa", "size"), con_asistencia=("asistio", "sum"),
                                  total_asistentes=("total_asistentes", "sum"))
calculado["sin_asistencia"] = calculado["total_empresas"] - calculado["con_asistencia"]
assert resumen.sort_index().equals(calculado[resumen.columns].sort_index()), (resumen, calculado)

# Tabla de la derecha
sin_asist = crudo.loc[crudo[7].astype("string").str.fullmatch(r"EMP-\d{4}", na=False), ["kam", 7, 9]]
assert set(sin_asist[9]) == {"Pendiente Convocatoria"}
assert set(zip(sin_asist["kam"], sin_asist[7])) == set(df.loc[~df["asistio"], ["kam", "id_empresa"]].itertuples(index=False, name=None))
resumen

## 4. Guardar
`utf-8-sig` para que Excel y Looker Studio lean bien las tildes. En Looker:
- % de empresas con asistencia por KAM = promedio de `asistio`.
- Empresas a convocar = filtro `asistio = False`.

In [ ]:
df = df[["kam", "id_empresa", "asistio", "eventos_asistidos", "total_asistentes"]]
salida = LIMPIOS / "09_asistencia_por_kam_limpio.csv"
df.to_csv(salida, index=False, encoding="utf-8-sig")

assert pd.read_csv(salida).shape == (460, 5)
salida.name